# Part III — Prediction Objective and Chronological Design

FNCE 2370 — Team 7 | Predicting Corporate Distress Risk

Assign splits by forecast outcome month, verify initial preprocessing, and export the common raw-feature modeling panel. Final refitting is disabled until the research design is frozen.

Run from the repository root (or its `notebooks/` subdirectory). See `README.md` for execution instructions. Source construction conventions are documented following the TA clarification of November 24, 2026; see `README.md` for remaining verification limits.

## Prediction objective

The required pooled dynamic logit estimates the conditional probability of a first recorded failure in the next calendar month, using the TA-confirmed narrower CRSP delisting-based definition (codes 550, 552, 560, 561, 574, 580, 584):

$$P(Y_{i,t+1}=1\mid\mathcal I_t,A_{i,t}=1)=\Lambda(\alpha+X_{i,t}^{\top}\beta),\qquad \Lambda(z)=\frac{1}{1+e^{-z}}.$$

Here $A_{i,t}$ denotes at-risk status and $X_{i,t}$ contains the eight CHS predictors observed as of month $t$, under the TA-confirmed upstream accounting availability rules described in Part II. Individual release dates remain unverified. This is a monthly hazard, not cumulative multimonth failure risk or a stock-return prediction. The benchmark yields probabilities (and hence rankings); score-only extensions require development-based calibration before being interpreted as probabilities. Model fitting and evaluation belong to Parts IV onward.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"
tables_dir = project_root / "outputs" / "tables"
src_dir = project_root / "src"

assert (processed_dir / "part_II_features.parquet").exists()
assert (tables_dir / "feature_manifest.csv").exists()
assert (tables_dir / "preprocessing_params_initial.csv").exists()
assert (src_dir / "preprocessing.py").exists()
sys.path.insert(0, str(src_dir))

from preprocessing import fit_preprocessor, transform_features

TRAIN_END = pd.Timestamp("1995-12-31")
VALIDATION_END = pd.Timestamp("2003-12-31")
TEST_END = pd.Timestamp("2024-12-31")
PERIODS = ["train", "validation", "test"]
KEYS = ["permno", "feature_month"]

## 1. Load the frozen feature manifest and Part II panel

In [2]:
manifest = pd.read_csv(tables_dir / "feature_manifest.csv")

# Explicit conversion protects against string-valued boolean columns.
selected_flag = manifest["selected"].astype(str).str.lower()
assert selected_flag.isin(["true", "false"]).all()
selected = manifest.loc[selected_flag.eq("true")]

CHS_FEATURES = selected.loc[selected["source"].eq("CHS"), "feature"].tolist()
GLOBAL_FEATURES = selected.loc[
    selected["source"].eq("Global Characteristics"), "feature"
].tolist()
FEATURES = CHS_FEATURES + GLOBAL_FEATURES

assert len(CHS_FEATURES) == 8
assert len(GLOBAL_FEATURES) == 37
assert len(set(FEATURES)) == 45

df = pd.read_parquet(processed_dir / "part_II_features.parquet")

assert set(FEATURES).issubset(df.columns)
assert not df.duplicated(KEYS).any()
assert df[KEYS].notna().all().all()
for column in ["month_end", "feature_month", "forecast_month"]:
    df[column] = pd.to_datetime(df[column], errors="raise")
    assert df[column].notna().all()
    assert df[column].dt.is_month_end.all()

assert df["feature_month"].eq(df["month_end"]).all()
assert df["forecast_month"].eq(
    df["feature_month"] + pd.offsets.MonthEnd(1)
).all()
assert df["forecast_month"].le(TEST_END).all()
assert df["target"].dropna().isin([0, 1]).all()
assert df["labeled"].notna().all()
assert df["baseline_eligible"].notna().all()
assert df["labeled"].eq(df["target"].notna()).all()

# Verify eligibility without filling missing features or outcomes.
required = [f for f in CHS_FEATURES if f != "sigma"]
eligible_check = np.ones(len(df), dtype=bool)
for f in required:
    eligible_check &= np.isfinite(
        pd.to_numeric(df[f], errors="raise").to_numpy(dtype=float, na_value=np.nan)
    )
assert np.array_equal(eligible_check, df["baseline_eligible"].to_numpy(dtype=bool))

print("Input shape:", df.shape)
print("Predictors:", len(FEATURES))
print("Input checks passed.")

Input shape: (3416761, 54)
Predictors: 45
Input checks passed.


## 2. Assign chronological splits and establish eligibility

Split on the outcome/forecast month, not the feature month, to align evaluation with the required January 2004–December 2024 window and keep training outcomes before subsequent forecasts. Initial training starts in May 1966 after baseline feature restrictions, rather than February 1963 in the broader panel. For example, December 2003 features predict January 2004 and belong to test. No random split or random k-fold cross-validation is used.

In [3]:
forecast = df["forecast_month"]
df["split"] = pd.Categorical(
    np.select(
        [forecast.le(TRAIN_END), forecast.le(VALIDATION_END)],
        ["train", "validation"],
        default="test"
    ),
    categories=PERIODS,
    ordered=True
)

common = df["baseline_eligible"].astype(bool)
labeled = df["labeled"].astype(bool)

train_mask = common & labeled & df["split"].eq("train")
validation_mask = common & labeled & df["split"].eq("validation")
test_mask = common & labeled & df["split"].eq("test")
development_mask = train_mask | validation_mask
test_forecast_mask = common & df["split"].eq("test")

assert train_mask.any() and validation_mask.any() and test_mask.any()
assert not (train_mask & validation_mask).any()
assert not (development_mask & test_mask).any()
assert (train_mask | validation_mask | test_mask).equals(common & labeled)
assert df.loc[train_mask, "forecast_month"].max() < df.loc[validation_mask, "forecast_month"].min()
assert df.loc[validation_mask, "forecast_month"].max() < df.loc[test_mask, "forecast_month"].min()
assert df.loc[development_mask, "forecast_month"].le(VALIDATION_END).all()

# Confirm both boundary cases are assigned correctly (if present).
assert df.loc[forecast.eq("1996-01-31"), "split"].eq("validation").all()
assert df.loc[forecast.eq("2004-01-31"), "split"].eq("test").all()

print("Initial training labeled:", int(train_mask.sum()))
print("Validation labeled:", int(validation_mask.sum()))
print("Final development labeled:", int(development_mask.sum()))
print("Final test labeled:", int(test_mask.sum()))
print("Final test forecast-eligible:", int(test_forecast_mask.sum()))
print("Chronological split checks passed.")

# Verify coverage of every required test forecast calendar month.
expected_test_months = pd.period_range("2004-01", "2024-12", freq="M")
observed_test_months = pd.PeriodIndex(
    df.loc[test_forecast_mask, "forecast_month"].dt.to_period("M").unique()
).sort_values()
missing_test_months = expected_test_months.difference(observed_test_months)
assert len(missing_test_months) == 0, f"Missing test months: {missing_test_months.tolist()}"
print("Test forecast months covered:", len(observed_test_months))

Initial training labeled: 1042054
Validation labeled: 594121
Final development labeled: 1636175
Final test labeled: 986920
Final test forecast-eligible: 986920
Chronological split checks passed.
Test forecast months covered: 252


## 3. Report sample coverage

These are descriptive counts for the already specified research design, not evidence for selecting models or preprocessing policies. Keep final-test outcomes out of model development. A positive security-month is not necessarily a unique corporate event; PERMCO-month counts are reported separately.

The original common sample has 1,042,054 labeled training rows, 594,121 validation rows and 986,920 test rows. The outcome is rare; preserve the natural class proportions in the shared panel. Any alternative sampling or weighting must be performed within model training, and evaluated on the unchanged common sample. Full-period outcome summaries are descriptive and must not guide selection.

In [4]:
def summarize_period(mask, period, universe):
    rows = df.loc[mask, [
        "permno", "permco", "feature_month", "forecast_month", "target"
    ]]
    known = rows["target"].notna()
    events = rows.loc[rows["target"].eq(1).fillna(False)]
    return {
        "universe": universe,
        "split": period,
        "feature_start": rows["feature_month"].min(),
        "feature_end": rows["feature_month"].max(),
        "forecast_start": rows["forecast_month"].min(),
        "forecast_end": rows["forecast_month"].max(),
        "forecast_observations": len(rows),
        "labeled_observations": int(known.sum()),
        "unlabeled_observations": int((~known).sum()),
        "unique_securities": rows["permno"].nunique(),
        "unique_companies": rows["permco"].nunique(),
        "labeled_unique_securities": rows.loc[known, "permno"].nunique(),
        "labeled_unique_companies": rows.loc[known, "permco"].nunique(),
        "positive_security_months": len(events),
        "unique_permco_month_failures": len(
            events.dropna(subset=["permco"])
            .drop_duplicates(["permco", "forecast_month"])
        ),
        "positive_rows_missing_permco": int(events["permco"].isna().sum()),
        "failure_rate_pct": rows["target"].mean() * 100
    }

records = []
for period in PERIODS:
    period_mask = df["split"].eq(period)
    records.append(summarize_period(period_mask, period, "Part I universe"))
    records.append(summarize_period(period_mask & common, period, "Common baseline-eligible"))

split_summary = pd.DataFrame(records)
display(split_summary)

common_summary = split_summary.loc[
    split_summary["universe"].eq("Common baseline-eligible")
].set_index("split")
assert common_summary["labeled_observations"].sum() == int((common & labeled).sum())
assert common_summary["forecast_observations"].sum() == int(common.sum())

,universe,split,feature_start,feature_end,forecast_start,forecast_end,forecast_observations,labeled_observations,unlabeled_observations,unique_securities,unique_companies,labeled_unique_securities,labeled_unique_companies,positive_security_months,unique_permco_month_failures,positive_rows_missing_permco,failure_rate_pct
0,Part I universe,train,1963-01-31,1995-11-30,1963-02-28,1995-12-31,1775631,1774654,977,17184,16896,17174,16886,2987,2979,0,0.168314
1,Common baseline-eligible,train,1966-04-30,1995-11-30,1966-05-31,1995-12-31,1042167,1042054,113,12848,12724,12847,12723,2207,2205,0,0.211793
2,Part I universe,validation,1995-12-31,2003-11-30,1996-01-31,2003-12-31,620643,620641,2,10887,10679,10887,10679,1981,1975,0,0.319186
3,Common baseline-eligible,validation,1995-12-31,2003-11-30,1996-01-31,2003-12-31,594123,594121,2,10624,10512,10624,10512,1925,1925,0,0.324008
4,Part I universe,test,2003-12-31,2024-11-30,2004-01-31,2024-12-31,1020487,1020487,0,10085,9906,10085,9906,1424,1417,0,0.139541
5,Common baseline-eligible,test,2003-12-31,2024-11-30,2004-01-31,2024-12-31,986920,986920,0,9620,9511,9620,9511,1335,1331,0,0.135269


## 4. Validate the initial preprocessing parameters

Fit only on initial labeled training data. Confirm the result agrees with the Part II parameter file. Transform small training and validation samples as a smoke test. Do not learn preprocessing parameters from validation or test; final refitting is a separate stage.

In [5]:
initial_params = pd.read_csv(
    tables_dir / "preprocessing_params_initial.csv", index_col="feature"
)
assert initial_params.index.tolist() == FEATURES

recomputed = fit_preprocessor(
    df.loc[train_mask, FEATURES],
    CHS_FEATURES,
    GLOBAL_FEATURES,
    lower=0.05,
    upper=0.95
)

pd.testing.assert_frame_equal(
    initial_params, recomputed,
    check_dtype=False, check_exact=False, rtol=1e-10, atol=1e-10
)

for mask in [train_mask, validation_mask]:
    raw_preview = df.loc[mask, FEATURES].head(1000)
    transformed = transform_features(raw_preview, initial_params)
    assert transformed.columns.tolist() == FEATURES
    assert transformed.index.equals(raw_preview.index)
    assert np.isfinite(transformed.to_numpy()).all()

print("Initial parameters reproduced using training only.")
print("Training and validation transformations passed.")

Initial parameters reproduced using training only.
Training and validation transformations passed.


## 5. Export the common modeling panel

The original Part II panel remains available for alternative sample rules and robustness tests. This new panel includes only baseline-eligible rows, retains unknown targets for prediction, and contains the chronological split. No features are permanently transformed. Identifiers, dates, targets and flags are metadata, not predictors.

In [6]:
metadata_cols = [
    "permno", "permco", "gvkey", "month_end", "feature_month",
    "forecast_month", "target", "labeled", "baseline_eligible", "split"
]
model_panel_path = processed_dir / "part_III_model_panel.parquet"

model_panel = df.loc[common, metadata_cols + FEATURES].copy()
assert not model_panel.duplicated(KEYS).any()
assert len(model_panel) == int(common.sum())
assert model_panel["labeled"].eq(model_panel["target"].notna()).all()

model_panel.to_parquet(model_panel_path, index=False, compression="snappy")

# Verify metadata on the written file.
import pyarrow.parquet as pq
written = pq.ParquetFile(model_panel_path).metadata
assert written.num_rows == len(model_panel)
assert written.num_columns == len(metadata_cols) + len(FEATURES)

split_summary.to_csv(tables_dir / "chronological_split_summary.csv", index=False)

print("Exported:", model_panel_path)
print("Rows:", written.num_rows)
print("Columns:", written.num_columns)
print("Size (GB):", round(model_panel_path.stat().st_size / 1e9, 3))
print("Exported:", tables_dir / "chronological_split_summary.csv")

Exported: /Users/tanirya/Desktop/UPenn/Wharton 7th Semester/FNCE 2370/Final Project/data/processed/part_III_model_panel.parquet
Rows: 2623210
Columns: 55
Size (GB): 0.653
Exported: /Users/tanirya/Desktop/UPenn/Wharton 7th Semester/FNCE 2370/Final Project/outputs/tables/chronological_split_summary.csv


## 6. Model handoff and final refitting protocol

**Model development:** use the `train` and `validation` rows with `labeled == True`. Select predictor specification, hyperparameters, threshold and calibration strategy without consulting test performance. The shared initial parameters can be used for the fixed initial train/validation split. For rolling-origin validation, refit preprocessing inside each training fold. Model-specific scaling must also be fitted on the corresponding estimation rows.

**Final refitting:** after the group freezes the research design, refit both preprocessing and model coefficients on the eligible labeled `train` + `validation` rows. Keep the chosen feature list and preprocessing method fixed. Use a calibration procedure and threshold selected exclusively from pre-2004 out-of-sample development predictions; do not fit calibration to the in-sample fitted values of the final model or to test outcomes. Document any calibration transfer limitations after refitting.

**Testing:** predict on all common `test` rows, including the unlabeled rows; calculate metrics only on the same labeled test observations across models. Store PERMNO, PERMCO, feature month, forecast month, target, split and model-specific probabilities. A score-only model needs development-based calibration before interpreting its output as a probability.

The following optional cell computes final preprocessing parameters only when the design is frozen. Leave it disabled during model selection. The model itself is estimated in Part IV or subsequent sections.

The fixed chronological design is required by the assignment; the selective-imputation eligibility and feature coverage screen are team choices. The final test is a fixed-design evaluation, with expanding/rolling refits permitted only as separately specified real-time exercises using already realized labels. `DESIGN_FROZEN` remains False until teammates have frozen their specifications.

The month-end boundary assumes that labels through December 2003 are available when the January 2004 forecast is formed. The file has no label-publication timestamps. For a strict real-time exercise, verify label availability or use a conservative extra outcome lag; do not claim that calendar alignment proves publication timing.

In [7]:
DESIGN_FROZEN = False  # Change only after all pre-2004 model choices are frozen.

if DESIGN_FROZEN:
    final_params = fit_preprocessor(
        df.loc[development_mask, FEATURES],
        CHS_FEATURES, GLOBAL_FEATURES,
        lower=0.05, upper=0.95
    )
    assert final_params.index.tolist() == FEATURES
    assert final_params["training_rows"].eq(int(development_mask.sum())).all()
    final_params.to_csv(tables_dir / "preprocessing_params_final.csv")
    print("Final preprocessing parameters saved; no test rows used.")
else:
    print("Final refitting deferred until model specifications are frozen.")

Final refitting deferred until model specifications are frozen.


## 7. Record the local execution environment

Record actual package versions for the final replication README. This cell does not change the model panel or any fitted parameters.

In [8]:
import platform
from importlib.metadata import version

environment = pd.Series({
    "python": platform.python_version(),
    **{name: version(name) for name in ["numpy", "pandas", "pyarrow", "matplotlib", "ipykernel"]}
}, name="version")
display(environment)
environment.to_csv(tables_dir / "replication_environment.csv", header=True)

python        3.11.5
numpy         1.24.3
pandas         2.0.3
pyarrow       11.0.0
matplotlib     3.7.2
ipykernel     6.25.0
Name: version, dtype: object